In [13]:
!pip install pgmpy


In [14]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination
from pgmpy.sampling import BayesianModelSampling
from pgmpy.factors.discrete import State

# 1. Build the network structure for Exercise 3
model = DiscreteBayesianNetwork([
    ('heatwave', 'overload'),
    ('heatwave', 'powercut'),
    ('overload', 'powercut'),
    ('powercut', 'cancel')
])

# Define CPDs with exact values from the assignment
cpd_h = TabularCPD(
    variable='heatwave', variable_card=2,
    values=[[0.4], [0.6]],
    state_names={'heatwave': ['no', 'yes']}
)

cpd_a = TabularCPD(
    variable='overload', variable_card=2,
    values=[[0.7, 0.9],
            [0.3, 0.1]],
    evidence=['heatwave'], evidence_card=[2],
    state_names={'overload': ['no', 'yes'], 'heatwave': ['no', 'yes']}
)

cpd_c = TabularCPD(
    variable='powercut', variable_card=2,
    values=[[0.3, 0.8, 0.5, 0.95],
            [0.7, 0.2, 0.5, 0.05]],
    evidence=['heatwave', 'overload'], evidence_card=[2, 2],
    state_names={
        'powercut': ['no', 'yes'],
        'heatwave': ['no', 'yes'],
        'overload': ['no', 'yes']
    }
)

cpd_l = TabularCPD(
    variable='cancel', variable_card=2,
    values=[[0.2, 0.9],
            [0.8, 0.1]],
    evidence=['powercut'], evidence_card=[2],
    state_names={'cancel': ['no', 'yes'], 'powercut': ['no', 'yes']}
)

# Add CPDs and verify model (should print True)
model.add_cpds(cpd_h, cpd_a, cpd_c, cpd_l)
print("Model check result:", model.check_model())

Model check result: True


In [15]:
# 2. Use Variable Elimination to find P(heatwave | cancel=yes)
infer = VariableElimination(model)
exact_result = infer.query(variables=['heatwave'], evidence={'cancel': 'yes'})
print("Exact Inference Result:")
print(exact_result)

Exact Inference Result:
+---------------+-----------------+
| heatwave      |   phi(heatwave) |
+===============+=================+
| heatwave(no)  |          0.3649 |
+---------------+-----------------+
| heatwave(yes) |          0.6351 |
+---------------+-----------------+


In [16]:
# 3. Use rejection sampling with 10,000 samples to estimate P(heatwave=yes | cancel=yes)
samples = BayesianModelSampling(model).rejection_sample(
    evidence=[State("cancel", "yes")],
    size=10000
)

prob_sampling = (samples['heatwave'] == 'yes').mean()
print(f"Rejection Sampling Estimate P(heatwave=yes | cancel=yes): {prob_sampling:.4f}")

  0%|          | 0/10000 [00:00<?, ?it/s]

Rejection Sampling Estimate P(heatwave=yes | cancel=yes): 0.6394


The exact inference result and the rejection sampling estimate are very close, though they are not identical due to the inherent randomness of sampling. As the sample size increases (such as to 10,000 samples), the approximation error decreases, bringing the sampling estimate closer to the true analytical probability.